In [11]:
import pandas as pd
import numpy as np
import os

MODEL_OUTPUT = r'..\..\..\outputs2020'
MODEL_INPUT = r'..\..\..\inputs2020'
MODEL_OUTPUT_2010 = r'C:\projects\IdahoSTDM\ITD_STDM\Models\BaseYear\Development\IdahoSTDM\ITDSTDM\outputs_2010'
MODEL_INPUT_2010 = r'C:\projects\IdahoSTDM\ITD_STDM\Models\BaseYear\Development\IdahoSTDM\ITDSTDM\inputs'

ldt_trips = pd.read_csv(os.path.join(MODEL_OUTPUT, 'LDTPersonTrips.csv'))
ldt_trips_2010 = pd.read_csv(os.path.join(MODEL_OUTPUT_2010, 'LDTPersonTrips.csv'))

taz = pd.read_csv(os.path.join(MODEL_INPUT, 'tazs.csv'))
taz_2010 = pd.read_csv(os.path.join(MODEL_INPUT_2010, 'tazs.csv'))
person_data = pd.read_csv(os.path.join(MODEL_OUTPUT, 'PersonData.csv')).merge(taz[['STDM_TAZ', 'State', 'County']], how = 'left', left_on = 'home_taz', right_on = 'STDM_TAZ')
person_data_2010 = pd.read_csv(os.path.join(MODEL_OUTPUT_2010, 'PersonData.csv')).merge(taz_2010[['STDM_TAZ', 'State', 'County']], how = 'left', left_on = 'home_taz', right_on = 'STDM_TAZ')


In [12]:
ldt_trips['Year'] = 2020
ldt_trips_2010['Year'] = 2010
person_data['Year'] = 2020
person_data_2010['Year'] = 2010
taz['Year'] = 2020
taz_2010['Year'] = 2010




ldt_trips_all = pd.concat([ldt_trips, ldt_trips_2010], axis=0)

person_data_all = pd.concat([person_data, person_data_2010], axis=0)

taz_all = pd.concat([taz, taz_2010], axis=0)

In [13]:
ldt_trips_all.groupby(['Year', 'tourMode']).size().reset_index(name='Records').pivot_table(index='tourMode', columns='Year', values='Records', fill_value=0).map(lambda x: f'{x:,.0f}')

Year,2010,2020
tourMode,,
AIR,"10,366","12,211"
AUTO,"88,604","111,676"


In [14]:
temp_df = ldt_trips_all.merge(person_data_all[['Year','HH_ID', 'memberID', 'State']], how='left', left_on=['Year','hhID', 'memberID'], right_on=['Year','HH_ID', 'memberID']).\
    groupby(['Year','State','tourMode']).size().reset_index(name='Records').\
        pivot_table(columns='Year', index=['State','tourMode'], values='Records', fill_value=0)

temp_total = pd.DataFrame(temp_df.sum(numeric_only=True)).T
multi_index = pd.MultiIndex.from_tuples([('Total', 'All')], names=['State', 'tourMode'])
temp_total.index = multi_index
temp_df = pd.concat([temp_df, temp_total])
temp_df['diff'] = temp_df[2020]-temp_df[2010]
temp_df['pct_diff'] = (temp_df['diff']/temp_df[2010])*100


temp_df.style.format({
    2010: '{:,.0f}',  # Add commas to VMT values
    2020: '{:,.0f}',
    'diff': '{:,.0f}',
    'pct_diff': '{:.2f}%'  # Format percentage with two decimal places
})

In [ ]:
temp_df = ldt_trips_all.merge(person_data_all[['Year','HH_ID', 'memberID', 'State']], how='left', left_on=['Year','hhID', 'memberID'], right_on=['Year','HH_ID', 'memberID']).groupby(['Year','State','tourMode']).distance.sum().reset_index(name='Records').pivot_table(columns='Year', index=['State','tourMode'], values='Records', fill_value=0)


temp_total = pd.DataFrame(temp_df.sum(numeric_only=True)).T
multi_index = pd.MultiIndex.from_tuples([('Total', 'All')], names=['State', 'tourMode'])
temp_total.index = multi_index
temp_df = pd.concat([temp_df, temp_total])
temp_df['diff'] = temp_df[2020]-temp_df[2010]
temp_df['pct_diff'] = (temp_df['diff']/temp_df[2010])*100


temp_df.style.format({
    2010: '{:,.0f}',  # Add commas to VMT values
    2020: '{:,.0f}',
    'diff': '{:,.0f}',
    'pct_diff': '{:.2f}%'  # Format percentage with two decimal places
})

Year                      2010        2020
State      tourMode                       
Idaho      AIR       1,142,645   1,519,333
           AUTO      9,395,133  12,676,414
Montana    AIR         179,319     322,664
           AUTO      1,884,503   2,578,574
Nevada     AIR          39,914     100,813
           AUTO        403,886     733,724
Oregon     AIR          52,205      42,639
           AUTO        264,153     324,902
Utah       AIR         105,686     149,947
           AUTO        841,124   1,103,743
Washington AIR         417,505     366,766
           AUTO      3,395,341   3,435,579
Wyoming    AIR          28,432      31,864
           AUTO        168,535     311,366

In [2]:
ldt_trips_joined = ldt_trips.merge(person_data, how = 'left', left_on = ['hhID', 'memberID'], right_on = ['HH_ID', 'memberID'])

In [3]:
person_data.groupby('State').agg(persons = ('HH_ID', 'count'))

,persons
State,
Idaho,1717984
Montana,392629
Nevada,86443
Oregon,62704
Utah,202495
Washington,494923
Wyoming,46900


In [4]:
ldt_trips_joined.groupby('State').agg(ldt_trips = ('hhID', 'count'))

,ldt_trips
State,
Idaho,71287
Montana,13722
Nevada,3757
Oregon,2073
Utah,6444
Washington,24846
Wyoming,1758


In [5]:
print(f"{ldt_trips.shape[0]:,.0f}")

123,887


In [6]:
print(f"{ldt_trips[ldt_trips['tourMode'] == 'AUTO'].shape[0]:,.0f}")

111,676


In [7]:
print(f"{ldt_trips['distance'].sum():,.0f}")

23,698,327


In [8]:
print(f"{ldt_trips[ldt_trips['tourMode'] == 'AUTO']['distance'].sum():,.0f}")

21,164,302


In [9]:
ldt_trips.distance.describe().map('{:,.2f}'.format)

count    123,887.00
mean         191.29
std          131.78
min            0.90
25%           94.57
50%          147.51
75%          264.21
max          880.48
Name: distance, dtype: object

In [10]:
ldt_trips_joined.groupby(['State', 'tourPurpose', 'tourMode']).agg(num_trips = ('hhID', 'count'))

num_trips
State      tourPurpose tourMode           
Idaho      HOUSEHOLD   AIR            1376
                       AUTO          16888
           OTHER       AIR            3037
                       AUTO          37819
           WORKRELATED AIR            1500
                       AUTO          10667
Montana    HOUSEHOLD   AIR              68
                       AUTO            941
           OTHER       AIR             845
                       AUTO           8320
           WORKRELATED AIR             461
                       AUTO           3087
Nevada     HOUSEHOLD   AIR             255
                       AUTO           1146
           OTHER       AIR             120
                       AUTO           1529
           WORKRELATED AIR              65
                       AUTO            642
Oregon     HOUSEHOLD   AIR               8
                       AUTO             84
           OTHER       AIR             138
                       AUTO           1367
           WORKRELATED AIR              24
                       AUTO            452
Utah       HOUSEHOLD   AIR             229
                       AUTO           1382
           OTHER       AIR             242
                       AUTO           3808
           WORKRELATED AIR             105
                       AUTO            678
Washington HOUSEHOLD   AIR            1440
                       AUTO           8051
           OTHER       AIR            1794
                       AUTO          10437
           WORKRELATED AIR             396
                       AUTO           2728
Wyoming    HOUSEHOLD   AIR              16
                       AUTO             84
           OTHER       AIR              43
                       AUTO           1243
           WORKRELATED AIR              49
                       AUTO            323